# Imports
Import required libraries for data processing, modeling, and evaluation.


In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
import joblib


# Load Data
Load the training, testing, and ground-truth RUL datasets.


In [2]:
columns = ['unit_number', 'time_in_cycles', 'op_setting_1', 'op_setting_2', 'op_setting_3'] + [f'sensor_{i}' for i in range(1, 22)]
train = pd.read_csv('data/train_FD001.txt', sep=r'\s+', header=None, names=columns)
test = pd.read_csv('data/test_FD001.txt', sep=r'\s+', header=None, names=columns)
rul_truth = pd.read_csv('data/RUL_FD001.txt', sep=r'\s+', header=None, names=['RUL'])


# Feature Engineering
Calculate training RUL (capped at 125), remove flat columns, and extract multi-window rolling stats and initial differences.


In [3]:
rul_cap = 125
max_cycle = train.groupby('unit_number')['time_in_cycles'].transform('max')
train['RUL'] = (max_cycle - train['time_in_cycles']).clip(upper=rul_cap)

candidate_cols = [c for c in train.columns if c.startswith('sensor_') or c.startswith('op_setting_')]
variances = train[candidate_cols].var()
flat_cols = variances[variances < 1e-4].index.tolist()
informative_sensors = [c for c in candidate_cols if c not in flat_cols]

window_sizes = [10, 20]
new_cols_train = {}
new_cols_test = {}

for w in window_sizes:
    for col in informative_sensors:
        new_cols_train[f'{col}_roll_mean_{w}'] = train.groupby('unit_number')[col].transform(lambda s: s.rolling(w, min_periods=1).mean())
        new_cols_test[f'{col}_roll_mean_{w}'] = test.groupby('unit_number')[col].transform(lambda s: s.rolling(w, min_periods=1).mean())
        
        new_cols_train[f'{col}_roll_std_{w}'] = train.groupby('unit_number')[col].transform(lambda s: s.rolling(w, min_periods=1).std().fillna(0))
        new_cols_test[f'{col}_roll_std_{w}'] = test.groupby('unit_number')[col].transform(lambda s: s.rolling(w, min_periods=1).std().fillna(0))

for col in informative_sensors:
    first_train = train.groupby('unit_number')[col].transform('first')
    first_test = test.groupby('unit_number')[col].transform('first')
    new_cols_train[f'{col}_init_diff'] = train[col] - first_train
    new_cols_test[f'{col}_init_diff'] = test[col] - first_test

train = pd.concat([train, pd.DataFrame(new_cols_train, index=train.index)], axis=1)
test = pd.concat([test, pd.DataFrame(new_cols_test, index=test.index)], axis=1)

# Time in cycles is removed to prevent timeline memorization / overfitting
feature_cols = informative_sensors + list(new_cols_train.keys())


# Scale and Prepare Train/Test Data
Standardize features on training data, extract final cycles for test engines, and scale test features.


In [4]:
scaler = StandardScaler()
X_train = scaler.fit_transform(train[feature_cols])
y_train = train['RUL'].values

test_last = test.groupby('unit_number').tail(1).copy()
rul_truth_idx = rul_truth.copy()
rul_truth_idx['unit_number'] = range(1, len(rul_truth_idx) + 1)
test_last = test_last.merge(rul_truth_idx, on='unit_number', how='left')

X_test = scaler.transform(test_last[feature_cols])
y_test = test_last['RUL'].clip(upper=rul_cap).values


# Hyperparameter Tuning
Tune hyperparameters for 6 regularized regression models using RandomizedSearchCV with GroupKFold cross-validation.


In [5]:
models_config = {
    'Random Forest': {
        'model': RandomForestRegressor(random_state=42, n_jobs=-1),
        'params': {'n_estimators': [100, 150], 'max_depth': [6, 8], 'min_samples_leaf': [5, 10]}
    },
    'XGBoost': {
        'model': XGBRegressor(random_state=42, n_jobs=-1, eval_metric='rmse'),
        'params': {'n_estimators': [100, 150], 'max_depth': [2, 3], 'learning_rate': [0.03, 0.05], 'reg_alpha': [1.0, 5.0], 'reg_lambda': [1.0, 5.0]}
    },
    'MLP': {
        'model': MLPRegressor(random_state=42, max_iter=200, early_stopping=True, validation_fraction=0.15, n_iter_no_change=10),
        'params': {'hidden_layer_sizes': [(32, 16), (16,)], 'alpha': [0.1, 1.0]}
    },
    'Gradient Boosting': {
        'model': GradientBoostingRegressor(random_state=42),
        'params': {'n_estimators': [100, 150], 'max_depth': [2, 3], 'learning_rate': [0.03, 0.05]}
    },
    'Extra Trees': {
        'model': ExtraTreesRegressor(random_state=42, n_jobs=-1),
        'params': {'n_estimators': [100, 150], 'max_depth': [6, 8], 'min_samples_leaf': [5, 10]}
    },
    'Ridge': {
        'model': Ridge(),
        'params': {'alpha': [10.0, 50.0, 100.0]}
    }
}

groups_train = train['unit_number'].values
cv = GroupKFold(n_splits=3)
tuning_results = {}

for name, config in models_config.items():
    search = RandomizedSearchCV(
        estimator=config['model'],
        param_distributions=config['params'],
        n_iter=3,
        scoring='neg_mean_absolute_error',
        cv=cv,
        random_state=42,
        n_jobs=-1
    )
    search.fit(X_train, y_train, groups=groups_train)
    best_est = search.best_estimator_
    
    # Calculate train MAE to check for overfitting
    train_pred = best_est.predict(X_train)
    train_mae = mean_absolute_error(y_train, train_pred)
    
    # Calculate test MAE
    test_pred = best_est.predict(X_test)
    test_mae = mean_absolute_error(y_test, test_pred)
    rmse = mean_squared_error(y_test, test_pred) ** 0.5
    r2 = r2_score(y_test, test_pred)
    
    tuning_results[name] = {'train_mae': train_mae, 'mae': test_mae, 'rmse': rmse, 'r2': r2, 'model': best_est}
    print(f'{name} - Train MAE: {train_mae:.2f}, Test MAE: {test_mae:.2f}, Gap: {test_mae - train_mae:.2f}, R2: {r2:.3f}')


Random Forest - Train MAE: 7.82, Test MAE: 9.64, Gap: 1.81, R2: 0.884
XGBoost - Train MAE: 8.32, Test MAE: 10.20, Gap: 1.88, R2: 0.874


c:\Users\ARYAN SINGH JADAUN\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


MLP - Train MAE: 7.15, Test MAE: 10.69, Gap: 3.55, R2: 0.875
Gradient Boosting - Train MAE: 9.48, Test MAE: 10.33, Gap: 0.85, R2: 0.870
Extra Trees - Train MAE: 8.86, Test MAE: 9.66, Gap: 0.79, R2: 0.890
Ridge - Train MAE: 12.96, Test MAE: 13.51, Gap: 0.55, R2: 0.830


# Model Selection
Compare all models on the test set and identify the best performer.


In [6]:
best_model_name = min(tuning_results, key=lambda name: tuning_results[name]['mae'])
best_model = tuning_results[best_model_name]['model']
print(f"Best Model: {best_model_name}")
print(f"Train MAE: {tuning_results[best_model_name]['train_mae']:.2f}")
print(f"Test MAE: {tuning_results[best_model_name]['mae']:.2f}")
print(f"Test RMSE: {tuning_results[best_model_name]['rmse']:.2f}")
print(f"Test R2: {tuning_results[best_model_name]['r2']:.3f}")


Best Model: Random Forest
Train MAE: 7.82
Test MAE: 9.64
Test RMSE: 13.62
Test R2: 0.884


# Save Best Model Pipeline
Package scaler, feature list, and best model, and serialize to disk.


In [7]:
pipeline_data = {
    'scaler': scaler,
    'flat_cols': flat_cols,
    'informative_sensors': informative_sensors,
    'feature_cols': feature_cols,
    'window_size': window_sizes,
    'rul_cap': rul_cap,
    'best_model_name': best_model_name,
    'model': best_model
}
joblib.dump(pipeline_data, 'best_model_pipeline.pkl')


['best_model_pipeline.pkl']

# Load and Test Pipeline
Load the serialized pipeline file and run a sample test prediction to verify correctness.


In [8]:
loaded = joblib.load('best_model_pipeline.pkl')
loaded_model = loaded['model']
loaded_scaler = loaded['scaler']
loaded_features = loaded['feature_cols']

sample_X = loaded_scaler.transform(test_last[loaded_features].head(5))
sample_preds = loaded_model.predict(sample_X)
print("Predictions:", np.round(sample_preds, 1))
print("Actual RUL:", y_test[:5])


Predictions: [120.3 114.2  62.3 101.5 105.3]
Actual RUL: [112  98  69  82  91]
